# Prediction carbon densities with random forest models

In [ ]:
# Libraries
import os, shutil, time
import pandas as pd
import dask.dataframe as dd
from dask_ml.wrappers import ParallelPostFit
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# Directories
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'
dir03 = '../paper_ecosystem_carbon/output/03_rf_predict/'

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

---

In [ ]:
# Initialize dask
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(queue='compute',
                       cores=12, #24
                       memory='256 GB',
                       account='bm0891', 
                       interface="ib0",
                       walltime='08:00:00',
                       local_directory='../dask/',
                       job_extra_directives=[
                           '-o ../dask/LOG_worker_%j.o',
                           '-e ../dask/LOG_worker_%j.o'])

# Scale dask clusters
cluster.scale(jobs=10)

# Configurate dashboard url
import dask
dask.config.config.get('distributed').get('dashboard') \
    .update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})

# Create client
from dask.distributed import Client
client = Client(cluster)
client

In [ ]:
# Get database
dd_base = dd.read_parquet(dir02 + 'dd_base_r001.parquet') \
    .repartition(partition_size='1000 MiB') \
    .persist()

In [ ]:
# Pandas dataframe with lat and lon values
df_latlon = dd_base[['lat', 'lon']].compute()

In [ ]:
# List with ecosystem types
list_etype = ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
              'temp_dnf', 'glob_shrub', 'glob_grass', 'glob_tundr', 
              'glob_mount', 'glob_crop', 'glob_past']

# List with covariates
vars_soilgrids2017 = ['soilgrids2017_bdricm', 'soilgrids2017_bdrlog',
                      'soilgrids2017_bdticm']

vars_soilgrids2020 = ['soilgrids2020_cec', 'soilgrids2020_cfvo', 
                      'soilgrids2020_clay', 'soilgrids2020_phh2o', 
                      'soilgrids2020_sand', 'soilgrids2020_silt']

vars_worldclim = ['worldclim_bio1', 'worldclim_bio3', 'worldclim_bio4', 
                  'worldclim_bio5', 'worldclim_bio6', 'worldclim_bio12', 
                  'worldclim_bio13', 'worldclim_bio14', 'worldclim_bio15', 
                  'worldclim_elev']

vars_geom90m = ['geom90m_convergence', 'geom90m_cti', 'geom90m_eastness',
               'geom90m_northness', 'geom90m_slope', 'geom90m_spi']

list_var_exp = [*vars_soilgrids2017, *vars_soilgrids2020, *vars_worldclim, 
                *vars_geom90m]

In [ ]:
# Create directory to store parameter files
# Directory path and string
dir_predicted_r001 = os.path.join(dir03, 'files_predicted_r001')

# Create directory
os.makedirs(dir_predicted_r001, exist_ok=True)

In [ ]:
def parallel_postfit_predict(X_train, y_train, X_predict, params):
    """
    Fits a ParallelPostFit RandomForestRegressor model with the given parameters
    and writes predictions to a Parquet file.

    Parameters:
        X_train: Training feature set
        y_train: Training target set
        X_predict: Features to make predictions on
        params: Dictionary of model parameters
        output_path: Path to save the output Parquet file
    """
    # Initialize and fit the model
    rfr = ParallelPostFit(
        estimator=RandomForestRegressor(
            min_samples_leaf=params['min_samples_leaf'],
            max_features=params['max_features'],
            n_estimators=params['n_estimators'],
            random_state=params['random_state'],
            n_jobs=-1
        ),
        scoring='r2'
    )
    rfr.fit(X_train, y_train)

    # Return predictions
    return rfr.predict(X_predict)


def random_forest_predict(ctype, etype):

    """Predict carbon values for 10 best performing models and
    export as parquet file"""
    
    # Get dataframe with parameters and ranks
    df_params = pd.read_csv(
         os.path.join(dir03, 'files_params_r001',
                      f'df_params_rank_{ctype}_{etype}.csv'))

    # Read training data
    df_train = dd_base[dd_base[f'train_{ctype}_{etype}'] == True]
    
    # Define x_train
    X_train = df_train[list_var_exp].to_dask_array(lengths=True).compute()
    
    # Define y_train (ensure consistency with 02_04_prep_database)
    if ctype == 'cveg':
        if etype in ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
                     'temp_dnf', 'glob_shrub']:
            cbase = 'cveg_biom'    
        if etype in ['glob_tundr', 'glob_mount']:
            cbase = 'cveg_tund'
        if etype in ['glob_grass', 'glob_past']:
            cbase = 'cveg_gras'
        if etype == 'glob_crop':
            cbase = 'cveg_crop'
    if ctype == 'csoil':
        cbase = 'csoil_000_100'
    
    y_train = df_train[cbase].to_dask_array(lengths=True)
        
    # Select columns of explanatory variables
    X_predict = dd_base[list_var_exp].to_dask_array(lengths=True).persist()

    df_latlonx = df_latlon
    
    # Predict and export values for best 10 models
    for rank in range(1, 11):
        # Filter parameters for the current rank
        df_params_rank = df_params[df_params.rank_test_score == rank]
        
        # Extract parameters as a dictionary
        params = {'min_samples_leaf': df_params_rank.min_samples_leaf.item(),
                  'max_features': df_params_rank.max_features.item(),
                  'n_estimators': df_params_rank.n_estimators.item(),
                  'random_state': df_params_rank.random_state.item()
                 }
        
        # Run the prediction
        rfr_predict = parallel_postfit_predict(
            X_train, y_train, X_predict, params)

        # Add as array to dataframe with lat and lon
        df_latlonx[f'rfr_{rank}'] = rfr_predict.compute()

    # Construct output file name
    output_file = os.path.join(dir03, 'files_predicted_r001', 
                               f'df_rfpred_{ctype}_{etype}.parquet')
    
    # Remove output_file if already exists
    if os.path.exists(output_file):
        os.remove(output_file)

    # Export as parquet file
    df_latlonx.to_parquet(output_file)

In [ ]:
# Predict cveg
for etype in list_etype:
    ctype = 'cveg'
    print(f'{ctype}_{etype}')
    %time random_forest_predict(ctype, etype)

In [ ]:
## Predict csoil
for etype in list_etype:
    ctype = 'csoil'
    print(f'{ctype}_{etype}')
    %time random_forest_predict(ctype, etype)

In [ ]:
cluster.close()